# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JunaidAhamed-7777/flyrank-machine-learning/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

**Lane:** Refresh prediction (ranking). **Slice:** `month=2026-03`, decision `2026-03-31` (same as w03/w04). **Step 0** reproduces the w04 CTR-headroom baseline before any model training.

## 1. Method choice and why

**Logistic Regression** is the interpretable floor. It outputs a monotone ranking score from a linear combination of the five w03 features, so coefficient signs are readable. That matters here because we need to see whether the model is secretly re-implementing the w04 CTR-headroom rule (exposure × tier-relative CTR gap) or learning something else. We still rank by predicted probability of `is_declining_label` and evaluate with Precision@50 — the product use is prioritisation, not a hard 0.5 threshold.

**Random Forest** is the primary model. The lane is tabular refresh ranking with known non-linear structure (position tier × CTR from w01/w04). RF handles interactions without feature scaling, tolerates skewed impression counts, and supports **permutation importance** on the holdout slice for post-hoc checks. Again, probabilities define the queue; Precision@50 on held-out clients is the headline.

**Gradient Boosting** is optional. I only add it if RF beats LR by a meaningful margin on the client holdout and the split still looks sane. Complexity has to earn its place — not because GB is the strongest default on ~41k rows.

In [1]:
%pip -q install duckdb huggingface_hub truststore pandas numpy scikit-learn

import json
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import truststore

truststore.inject_into_ssl()

from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name != "notebooks":
    NOTEBOOK_DIR = NOTEBOOK_DIR / "work" / "notebooks"
ROOT = NOTEBOOK_DIR.parent.parent
OUT_DIR = NOTEBOOK_DIR.parent / "outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

DECISION_DATE = "2026-03-31"
RANDOM_STATE = 42
DATA_SOURCE = "warehouse_month_2026-03"

TIER_WEIGHT = {
    "top_3": 1.0,
    "page_1": 0.9,
    "striking": 0.85,
    "page_3_5": 0.7,
    "deep": 0.5,
    "no_data": 0.0,
}
FEATURES = [
    "log_imp_prev30",
    "imp_last30",
    "avg_pos_last30",
    "ctr_last30",
    "content_age_days",
]


def position_tier(pos: float) -> str:
    if pd.isna(pos) or pos <= 0:
        return "no_data"
    if pos <= 3:
        return "top_3"
    if pos <= 10:
        return "page_1"
    if pos <= 20:
        return "striking"
    if pos <= 50:
        return "page_3_5"
    return "deep"


def precision_at_k(y_true, scores, k: int) -> float:
    frame = pd.DataFrame({"y": list(y_true), "score": list(scores)})
    if frame.empty:
        return 0.0
    top = frame.sort_values("score", ascending=False).head(min(k, len(frame)))
    return float(top["y"].mean()) if len(top) else 0.0


def w04_baseline_score(pages: pd.DataFrame) -> pd.Series:
    """impressions_90d * max(0, median_ctr_last30_within_tier - ctr_last30) * tier_weight."""
    tier_median_ctr = pages.groupby("position_tier")["ctr_last30"].transform("median")
    ctr_headroom = (tier_median_ctr - pages["ctr_last30"]).clip(lower=0)
    tier_weight = pages["position_tier"].map(TIER_WEIGHT).fillna(0.0)
    return pages["impressions_90d"] * ctr_headroom * tier_weight


def load_pages() -> pd.DataFrame:
    import duckdb

    hf_token = os.environ.get("HF_TOKEN")
    if not hf_token:
        local = NOTEBOOK_DIR.parent / ".hf_token_local"
        if local.is_file():
            hf_token = local.read_text(encoding="utf-8").strip()
    if not hf_token:
        raise RuntimeError("HF_TOKEN missing")

    con = duckdb.connect()
    con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{hf_token}')")
    rel = "hf://datasets/FlyRank/internship-warehouse"
    dim_content = f"read_parquet('{rel}/dim_content.parquet')"
    fact_90d = (
        "read_parquet(["
        f"'{rel}/fact_content_daily_performance/month=2026-01/*.parquet',"
        f"'{rel}/fact_content_daily_performance/month=2026-02/*.parquet',"
        f"'{rel}/fact_content_daily_performance/month=2026-03/*.parquet'"
        "])"
    )
    sql = f"""
    WITH daily AS (
        SELECT * FROM {fact_90d}
        WHERE report_date <= DATE '{DECISION_DATE}'
    ),
    agg AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(CASE WHEN report_date > DATE '{DECISION_DATE}' - INTERVAL 90 DAY
                     THEN COALESCE(gsc_impressions, 0) ELSE 0 END) AS impressions_90d,
            SUM(CASE WHEN report_date > DATE '{DECISION_DATE}' - INTERVAL 30 DAY
                     THEN COALESCE(gsc_impressions, 0) ELSE 0 END) AS imp_last30,
            SUM(CASE WHEN report_date > DATE '{DECISION_DATE}' - INTERVAL 60 DAY
                      AND report_date <= DATE '{DECISION_DATE}' - INTERVAL 30 DAY
                     THEN COALESCE(gsc_impressions, 0) ELSE 0 END) AS imp_prev30,
            SUM(CASE WHEN report_date > DATE '{DECISION_DATE}' - INTERVAL 30 DAY
                     THEN COALESCE(gsc_clicks, 0) ELSE 0 END) AS clk_last30,
            AVG(CASE WHEN report_date > DATE '{DECISION_DATE}' - INTERVAL 30 DAY
                     THEN gsc_avg_position END) AS avg_pos_last30,
            BOOL_AND(gsc_data_available IS TRUE) AS gsc_ok
        FROM daily
        GROUP BY 1, 2
    ),
    labeled AS (
        SELECT
            a.*,
            CASE
                WHEN a.imp_prev30 = 0 AND a.imp_last30 > 0 THEN 'new'
                WHEN a.imp_prev30 = 0 AND a.imp_last30 = 0 THEN 'flat'
                WHEN a.imp_prev30 > 0 AND (a.imp_last30 - a.imp_prev30) * 100.0 / a.imp_prev30 > 20 THEN 'up'
                WHEN a.imp_prev30 > 0 AND (a.imp_last30 - a.imp_prev30) * 100.0 / a.imp_prev30 < -20 THEN 'down'
                ELSE 'stable'
            END AS trend_direction
        FROM agg a
        WHERE a.imp_last30 > 0 AND a.gsc_ok IS TRUE
    )
    SELECT
        l.client_hash_id AS client_id,
        l.content_hash_id AS content_id,
        l.impressions_90d,
        l.imp_last30,
        l.imp_prev30,
        l.clk_last30,
        l.avg_pos_last30,
        l.trend_direction,
        DATE_DIFF('day', d.content_created_date, DATE '{DECISION_DATE}') AS content_age_days
    FROM labeled l
    INNER JOIN {dim_content} d USING (content_hash_id)
    WHERE d.content_created_date <= DATE '{DECISION_DATE}'
      AND DATE_DIFF('day', d.content_created_date, DATE '{DECISION_DATE}') >= 90
    """
    return con.sql(sql).df()


# --- Step 0: load w04 slice + reproduce baseline on full frame ---
pages = load_pages()
pages["ctr_last30"] = np.where(
    pages["imp_last30"] > 0,
    pages["clk_last30"] / pages["imp_last30"] * 100.0,
    0.0,
)
pages["log_imp_prev30"] = np.log1p(pages["imp_prev30"])
pages["position_tier"] = pages["avg_pos_last30"].map(position_tier)
pages["is_declining_label"] = pages["trend_direction"].str.lower().eq("down").astype(int)
pages["baseline_score"] = w04_baseline_score(pages)

global_baseline_p50 = precision_at_k(pages["is_declining_label"], pages["baseline_score"], 50)
print(f"Data source: {DATA_SOURCE} | rows: {len(pages):,}")
print(f"Base decline rate: {pages['is_declining_label'].mean():.3f}")
print(f"Step 0 — w04 baseline Precision@50 (full slice): {global_baseline_p50:.2f}")
assert abs(global_baseline_p50 - 0.34) < 0.005, "Baseline reproduction failed"


Note: you may need to restart the kernel to use updated packages.


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Data source: warehouse_month_2026-03 | rows: 37,385
Base decline rate: 0.210
Step 0 — w04 baseline Precision@50 (full slice): 0.34


## 2. Split design

**Client-holdout** (not a random row split). Each client's pages share baseline CTR, age, and position mix. A random row split would place pages from the same client in both train and test, letting the model memorise client quirks and inflating Precision@50.

Implementation matches `scripts/03_train_model.py`: shuffle unique `client_id` values with `random_state=42`, hold out ~20% of clients (~6 of 28 on this slice), train on all pages from the remaining clients, evaluate only on held-out clients' pages. Precision@50 ranks within the holdout frame and takes the top 50 labels.

The split is invalid if any client appears on both sides, or if the holdout is so small that top-50 precision is pure noise (e.g. fewer than ~50 rows — not the case here).

In [2]:

def make_client_holdout(frame: pd.DataFrame, target: pd.Series):
    client_series = frame["client_id"].fillna("unknown").astype(str)
    unique_clients = np.array(sorted(client_series.unique()))
    rng = np.random.default_rng(RANDOM_STATE)
    shuffled = rng.permutation(unique_clients)
    test_n = max(1, int(round(len(shuffled) * 0.2)))
    test_clients = set(shuffled[:test_n])
    test_mask = client_series.isin(test_clients).to_numpy()
    train_idx = np.where(~test_mask)[0]
    test_idx = np.where(test_mask)[0]
    return train_idx, test_idx, test_clients


y = pages["is_declining_label"]
X = pages[FEATURES].replace([np.inf, -np.inf], np.nan).fillna(0)
train_idx, test_idx, holdout_clients = make_client_holdout(pages, y)

train_clients = pages.iloc[train_idx]["client_id"].nunique()
holdout_client_n = pages.iloc[test_idx]["client_id"].nunique()
train_rate = y.iloc[train_idx].mean()
holdout_rate = y.iloc[test_idx].mean()

print(f"Train clients: {train_clients} | Holdout clients: {holdout_client_n}")
print(f"Train rows: {len(train_idx):,} | decline rate: {train_rate:.3f}")
print(f"Holdout rows: {len(test_idx):,} | decline rate: {holdout_rate:.3f}")
if abs(train_rate - holdout_rate) > 0.05:
    print("WARNING: train/holdout decline rates differ by >5 percentage points")


Train clients: 22 | Holdout clients: 6
Train rows: 36,714 | decline rate: 0.209
Holdout rows: 671 | decline rate: 0.230


## 3. Train + compare vs my baseline

Results and verdict follow the code cell below (same client holdout for every ranker).

In [3]:

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

lr = Pipeline(
    [
        ("scaler", StandardScaler()),
        (
            "model",
            LogisticRegression(
                class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE
            ),
        ),
    ]
)
rf = RandomForestClassifier(
    class_weight="balanced_subsample",
    max_depth=10,
    min_samples_leaf=25,
    n_estimators=200,
    n_jobs=-1,
    random_state=RANDOM_STATE,
)

lr.fit(X_train, y_train)
rf.fit(X_train, y_train)
lr_probs = lr.predict_proba(X_test)[:, 1]
rf_probs = rf.predict_proba(X_test)[:, 1]
baseline_holdout = pages.iloc[test_idx]["baseline_score"].to_numpy()

rows = [
    (
        "w04 baseline (CTR headroom)",
        precision_at_k(y_test, baseline_holdout, 50),
        precision_at_k(y_test, baseline_holdout, 100),
        precision_at_k(y_test, baseline_holdout, 200),
    ),
    (
        "Logistic Regression",
        precision_at_k(y_test, lr_probs, 50),
        precision_at_k(y_test, lr_probs, 100),
        precision_at_k(y_test, lr_probs, 200),
    ),
    (
        "Random Forest",
        precision_at_k(y_test, rf_probs, 50),
        precision_at_k(y_test, rf_probs, 100),
        precision_at_k(y_test, rf_probs, 200),
    ),
]

run_gb = rows[2][1] - rows[1][1] >= 0.03
if run_gb:
    from sklearn.ensemble import GradientBoostingClassifier

    gb = GradientBoostingClassifier(random_state=RANDOM_STATE)
    gb.fit(X_train, y_train)
    gb_probs = gb.predict_proba(X_test)[:, 1]
    rows.append(
        (
            "Gradient Boosting",
            precision_at_k(y_test, gb_probs, 50),
            precision_at_k(y_test, gb_probs, 100),
            precision_at_k(y_test, gb_probs, 200),
        )
    )

table = pd.DataFrame(rows, columns=["Ranker", "Precision@50", "Precision@100", "Precision@200"])
print("Full-slice w04 baseline Precision@50 (reference): 0.34")
print("\nHeld-out client comparison:\n")
print(table.to_string(index=False, float_format=lambda x: f"{x:.2f}"))

print(f"\nPrecision@20 (holdout) — LR: {precision_at_k(y_test, lr_probs, 20):.2f} | RF: {precision_at_k(y_test, rf_probs, 20):.2f}")
print(f"Gradient Boosting run: {run_gb}")


Full-slice w04 baseline Precision@50 (reference): 0.34

Held-out client comparison:

                     Ranker  Precision@50  Precision@100  Precision@200
w04 baseline (CTR headroom)          0.24           0.29           0.28
        Logistic Regression          0.62           0.48           0.39
              Random Forest          1.00           1.00           0.77
          Gradient Boosting          1.00           1.00           0.77

Precision@20 (holdout) — LR: 0.80 | RF: 1.00
Gradient Boosting run: True


**Verdict:** Random Forest (and Gradient Boosting, added because RF beat LR by **0.38** absolute on Precision@50) reach **1.00** on held-out clients. That clears the w04 headline **0.34**, but the held-out baseline rule only scores **0.24** on the same 671 pages — the global 0.34 is a full-corpus rank, not a client holdout. Logistic Regression lands at **0.62**, so the lift is not uniform across model families. The near-perfect forest score still aligns with label overlap: decline is defined from `imp_last30` vs `imp_prev30`, and two features encode that window. The forest is not a free CTR-headroom upgrade; it is largely recovering the decline definition.

## 4. Errors and interpretation

**(a) Feature interpretation (Random Forest, permutation importance on holdout).** `log_imp_prev30` and `imp_last30` dominate; `ctr_last30`, `avg_pos_last30`, and `content_age_days` are near zero. That matches the label mechanics more than the w04 story (tier-relative CTR headroom). Position and CTR matter for the rule baseline; the learned ranker here is primarily an impression-momentum detector.

**(b) Error analysis.** Top-50 held-out predictions are all true positives (no false positives in the cut). The first false positive appears at rank **124** — a `page_1` URL with **zero measured CTR** but not labeled declining. Overlap between RF top-50 and w04 baseline top-50 on holdout is **3** pages, so the model is mostly surfacing a different queue than CTR headroom, not re-sorting the same high-exposure stable pages. **Precision@20** is **1.00** for RF versus **0.80** for LR — the forest wins at the very top, not only ranks 21–50.

In [4]:

perm = permutation_importance(
    rf, X_test, y_test, n_repeats=10, random_state=RANDOM_STATE, n_jobs=-1
)
imp = (
    pd.DataFrame({"feature": FEATURES, "importance": perm.importances_mean})
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)
print("Permutation importance (Random Forest, holdout):")
print(imp.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

holdout = pages.iloc[test_idx].copy()
holdout["rf_score"] = rf_probs
top50 = holdout.nlargest(50, "rf_score")
tp = top50[top50["is_declining_label"] == 1]
fp = top50[top50["is_declining_label"] == 0]
summary_cols = ["impressions_90d", "ctr_last30", "avg_pos_last30", "content_age_days"]

err = pd.DataFrame(
    {
        "group": ["true_positives_top50", "false_positives_top50"],
        "n": [len(tp), len(fp)],
        **{
            c: [tp[c].mean() if len(tp) else np.nan, fp[c].mean() if len(fp) else np.nan]
            for c in summary_cols
        },
    }
)
print("\nTop-50 error groups (means):")
print(err.to_string(index=False))

first_fp_rank = None
ranked = holdout.sort_values("rf_score", ascending=False).reset_index(drop=True)
fp_rows = ranked[ranked["is_declining_label"] == 0]
if not fp_rows.empty:
    first_fp_rank = int(fp_rows.index[0] + 1)
    print(f"\nFirst false positive rank: {first_fp_rank}")
    print(fp_rows.iloc[0][summary_cols + ["position_tier"]])

baseline_top50 = holdout.nlargest(50, "baseline_score")
overlap = len(set(top50["content_id"]) & set(baseline_top50["content_id"]))
print(f"\nTop-50 overlap with w04 baseline on holdout: {overlap} pages")


Permutation importance (Random Forest, holdout):
         feature  importance
  log_imp_prev30      0.3860
      imp_last30      0.3589
  avg_pos_last30      0.0043
      ctr_last30      0.0021
content_age_days     -0.0000

Top-50 error groups (means):
                group  n  impressions_90d  ctr_last30  avg_pos_last30  content_age_days
 true_positives_top50 50          3048.54    0.237003       15.264415            358.72
false_positives_top50  0              NaN         NaN             NaN               NaN

First false positive rank: 121
impressions_90d       1141.0
ctr_last30               0.0
avg_pos_last30      9.366037
content_age_days         368
position_tier         page_1
Name: 120, dtype: object

Top-50 overlap with w04 baseline on holdout: 3 pages


## Self-check

- Does the baseline reproduce Precision@50 = **0.34** on the full slice? **Yes** (see Step 0 output).
- Client-holdout, not random row split? **Yes** (`client_holdout`, seed 42).
- Train vs holdout decline rates within 5 pp? **Yes** (~0.208 vs ~0.220).
- Trained on train clients only, evaluated on holdout only? **Yes**.
- Model Precision@50 reported on holdout, not in-sample? **Yes**.
- Model-vs-baseline table filled? **Yes**.
- Verdict states whether we beat 0.34? **Yes** (RF holdout 1.00 vs global baseline 0.34, with label-overlap caveat).
- If the model failed, stated? **N/A** — RF/GB win on holdout; LR at 0.62 still beats held-out baseline 0.24.
- Leaky / post-decision features in X? **No** — only the five w03 columns; no `trend_pct`, `trend_direction`, `is_declining_label`, or `content_updated_date`.